# 08. Proposed Hybrid Framework: GARCH + Machine Learning
### Architecture
Combines econometric structural time-series information ($\hat{\sigma}_{GARCH, t}$) with non-linear feature representations:
$$\hat{\sigma}_{t, t+k}^{Hybrid} = f_{ML}\left(\hat{\sigma}_{GARCH, t}, \mathbf{x}_t^{returns}, \mathbf{x}_t^{volatility}, \mathbf{x}_t^{range}, \mathbf{x}_t^{volume}\right)$$


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.features import build_feature_dataset
from src.hybrid import HybridGARCHMLModel
from src.validation import chronological_split

df = pd.read_csv("../data/processed/nifty50_daily_processed.csv", parse_dates=["Date"], index_col="Date")
feat_df = build_feature_dataset(df, target_horizon=5)

target_col = "target_rv_5d"
feature_cols = [c for c in feat_df.columns if c not in [target_col, "Close", "log_return", "simple_return"]]
train_df, val_df, test_df = chronological_split(feat_df, train_end="2018-12-31", val_end="2020-12-31")

hybrid = HybridGARCHMLModel(base_learner="xgboost")
hybrid.fit(train_df[feature_cols], train_df[target_col], train_df["log_return"])

print("Hybrid Model Fitted Successfully.")
print("Top Feature Importances in Hybrid Architecture:")
print(hybrid.get_feature_importances().head(8))
